# 05d: Computational efficiency: CPU benchmark (on Google Colab CPU runtime)

Paper section: *Computational Efficiency*, paragraph *CPU*. We ran this notebook on a Google Colab standard CPU runtime, which is one Intel Xeon core at 2.2 GHz (2 vCPU, torch with 2 threads), 12 GB of memory, torch 2.14.1+cpu.

In [ ]:
import json
import os
import subprocess
import sys
import threading
import time
from pathlib import Path

IN_COLAB = Path("/content").is_dir()
WORK = Path(os.environ.get("BENCH_DIR", "/content" if IN_COLAB else "runs/efficiency"))
SOURCE = WORK if IN_COLAB else Path("data/efficiency")
WORK.mkdir(parents=True, exist_ok=True)
SMOKE = False

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "pyect", "trimesh"], check=True)

import numpy as np
import torch
from pyect import WECT, sample_directions_2d, sample_directions_3d

from diwect import DEFAULT_CHUNK_SIZE, DIWECT

torch.set_num_threads(os.cpu_count() or 1)
HEIGHTS = (17, 65, 257)
TARGETS = [(h, 1e-2) for h in HEIGHTS] + [(65, 1e-3)]
WECT_BATCH_BYTES = 2 * 2 ** 30
WECT_TIME_CAP = 2 * 3600.0
RESULTS = WORK / "results_cpu.jsonl"

## GPU results: reference values and the numbers of directions to time

In [ ]:
references = {}
for line in open(SOURCE / "results_diwect.jsonl"):
    r = json.loads(line)
    if r["kind"] == "diwect" and r["dtype"] == "float64" and "values" in r:
        references[r["complex"], r["h"]] = torch.tensor(r["values"], dtype=torch.float64)

gpu_wect = {}
for line in open(SOURCE / "results.jsonl"):
    r = json.loads(line)
    if r["kind"] == "wect" and r["sampling"] == "grid":
        gpu_wect.setdefault(r["complex"], []).append(r)


def needed_d(name):
    rows = sorted(gpu_wect.get(name, []), key=lambda r: r["d"])
    ds = set()
    for h, tol in TARGETS:
        hit = next((r for r in rows if r["rel_err"][str(h)] <= tol), None)
        if hit:
            ds.add(hit["d"])
    return sorted(ds)

## Resume and measurement helpers

In [ ]:
def key(r):
    return (r["kind"], r.get("complex"), r.get("h"), r.get("dtype"), r.get("d"))


done = set()
if RESULTS.exists():
    done = {key(json.loads(line)) for line in open(RESULTS) if line.strip()}
OUT = open(RESULTS, "a")


def emit(**rec):
    OUT.write(json.dumps(rec) + "\n")
    OUT.flush()
    print(json.dumps(rec), flush=True)


def rss():
    try:
        with open("/proc/self/statm") as f:
            return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE")
    except OSError:
        return 0


class PeakRSS:
    """Peak resident set size above the level at entry sampled in a thread"""

    def __enter__(self):
        self.base = self.peak = rss()
        self.stop = threading.Event()
        self.thread = threading.Thread(target=self.run, daemon=True)
        self.thread.start()
        return self

    def run(self):
        while not self.stop.wait(0.005):
            self.peak = max(self.peak, rss())

    def __exit__(self, *exc):
        self.stop.set()
        self.thread.join()
        self.peak = max(self.peak, rss())
        self.extra = self.peak - self.base


def measure(fn):
    """(seconds, peak extra RSS bytes, output, number of timed runs)."""
    with PeakRSS() as m:
        start = time.perf_counter()
        out = fn()
        t = time.perf_counter() - start
    if t >= 5:
        return t, m.extra, out, 1
    reps = 3 if t > 0.5 else 5
    times = []
    with PeakRSS() as m:
        for _ in range(reps):
            start = time.perf_counter()
            out = fn()
            times.append(time.perf_counter() - start)
    return float(np.median(times)), m.extra, out, reps


def rel_err(approx, exact):
    return float((approx - exact).abs().max() / exact.abs().max())


data = np.load(SOURCE / "inputs.npz")
index = json.loads(bytes(data["index"]).decode())


def load(name, dtype):
    return [(torch.as_tensor(data[f"{name}/{k}/x"]).to(dtype if k == 0 else torch.long),
             torch.as_tensor(data[f"{name}/{k}/w"]).to(dtype))
            for k in range(len(index[name]["cells"]))]


def wect_mean(cx, dirs, h, batch):
    """Average of pyECT's WECT over dirs, in batches of `batch` directions"""
    acc = torch.zeros(h, dtype=torch.float64)
    for i in range(0, dirs.size(0), batch):
        acc += WECT(dirs[i:i + batch], h)(cx).double().sum(0)
    return acc / dirs.size(0)


def batch_size(name):
    per_dir = max((k + 2) * n for k, n in enumerate(index[name]["cells"])) * 8 * 1.5
    return max(1, int(WECT_BATCH_BYTES // per_dir))


def common(name):
    m = index[name]
    return dict(complex=name, cells=m["total_cells"], ctype=m["kind"], dim=m["dim"], source=m["source"])

## The benchmark

In [ ]:
if not done:
    import platform
    cpu = subprocess.run("lscpu | grep 'Model name'", shell=True, capture_output=True, text=True).stdout
    emit(kind="env", torch=torch.__version__, device="cpu", cpu=cpu.split(":")[-1].strip() or platform.processor(),
         cpu_count=os.cpu_count(), threads=torch.get_num_threads(), chunk=DEFAULT_CHUNK_SIZE)

order = sorted(index, key=lambda n: index[n]["total_cells"])
if SMOKE:
    order = [n for n in order if index[n]["total_cells"] < 10000]

# --- DIWECT, all complexes first ---------------------------------------------- #
for name in order:
    dim, ref_name = index[name]["dim"], name.replace("cubical", "freudenthal")
    for dtype in (torch.float64, torch.float32):
        cx = None
        for h in HEIGHTS:
            dt = str(dtype)[6:]
            if ("diwect", name, h, dt, None) in done:
                continue
            cx = cx or load(name, dtype)
            mod = DIWECT(h, normalize=True, dtype=dtype)
            t, mem, out, reps = measure(lambda: mod(cx))
            emit(kind="diwect", h=h, dtype=dt, seconds=t, peak_rss_bytes=mem, reps=reps,
                 rel_err=rel_err(out.double(), references[ref_name, h]),
                 chunk_size=DEFAULT_CHUNK_SIZE[dim], **common(name))
        del cx

# --- WECT averaged over pyECT's directions at the desired d -------------------- #
for name in order:
    dim, ref_name = index[name]["dim"], name.replace("cubical", "freudenthal")
    sample = sample_directions_2d if dim == 2 else sample_directions_3d
    hmax, batch = max(HEIGHTS), batch_size(name)
    cx, last = None, None
    for d in needed_d(name):
        if ("wect", name, None, None, d) in done:
            continue
        if last and last[1] * d / last[0] > WECT_TIME_CAP:
            emit(kind="wect", d=d, skipped=True, predicted_seconds=last[1] * d / last[0], **common(name))
            continue
        cx = cx or load(name, torch.float32)
        dirs, b = sample(d), min(batch, d)
        t, mem, out, reps = measure(lambda: wect_mean(cx, dirs, hmax, b))
        errs = {str(h): rel_err(out[::(hmax - 1) // (h - 1)], references[ref_name, h]) for h in HEIGHTS}
        emit(kind="wect", sampling="grid", d=d, batch=b, seconds=t, peak_rss_bytes=mem, reps=reps,
             rel_err=errs, **common(name))
        last = (d, t)
    for h in (HEIGHTS[0], HEIGHTS[-1]):
        if ("wect_h_check", name, h, None, None) in done:
            continue
        cx = cx or load(name, torch.float32)
        dirs = sample(64)
        t, _, _, _ = measure(lambda: wect_mean(cx, dirs, h, min(batch, 64)))
        emit(kind="wect_h_check", d=64, h=h, seconds=t, **common(name))
    del cx

emit(kind="done")
OUT.close()